# Task 3.1 — Two-Component Ablation Study

In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import sys; sys.path.insert(0, '.')
from smm_core import *

np.random.seed(42); rng = np.random.RandomState(42)

# Dataset
N_PER_CLASS=100; N_SAMPLES=20; D=2
distributions, labels = [], []
for _ in range(N_PER_CLASS):
    distributions.append(rng.multivariate_normal(rng.randn(D)*0.6, np.array([[1.8,0.05],[0.05,0.25]]), N_SAMPLES))
    labels.append(+1)
for _ in range(N_PER_CLASS):
    distributions.append(rng.multivariate_normal(rng.randn(D)*0.6, np.array([[0.25,0.05],[0.05,1.8]]), N_SAMPLES))
    labels.append(-1)
labels = np.array(labels)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def run_smm(distributions, labels, gamma=0.5, kernel='rbf', level2='precomputed', C=1.0):
    scores = []
    for train_idx, test_idx in kf.split(np.arange(len(distributions)), labels):
        td = [distributions[i] for i in train_idx]
        ted = [distributions[i] for i in test_idx]
        Ktr = build_smm_kernel_matrix(td, gamma=gamma, kernel=kernel)
        Kte = build_smm_kernel_matrix_train_test(ted, td, gamma=gamma, kernel=kernel)
        clf = SVC(kernel=level2, C=C)
        clf.fit(Ktr, labels[train_idx])
        scores.append(accuracy_score(labels[test_idx], clf.predict(Kte)))
    return np.array(scores)

print("Dataset ready, ablation functions defined.")

Dataset ready, ablation functions defined.


## Component 1: Embedding Kernel (RBF → Linear)

**Role in the full method:**
The embedding kernel k is the base kernel used to construct the mean embedding μ_P = ∫ k(x,·) dP(x) (Eq. 1). When k is an RBF kernel, the expected kernel K(P_i, P_j) = E[exp(-γ||x-z||²)] captures the covariance structure of both P_i and P_j (see Table 1 of the paper). Replacing it with a linear kernel k(x,z) = ⟨x,z⟩ reduces K(P_i,P_j) to E_{x~Pi}[x]ᵀ E_{z~Pj}[z] = ⟨μᵢ, μⱼ⟩, which is merely the dot product of the distribution means — equivalent to an SVM on means.


In [ ]:
smm_full = run_smm(distributions, labels, gamma=0.5, kernel='rbf')
smm_abl1 = run_smm(distributions, labels, gamma=0.5, kernel='linear')

print(f"Full SMM (RBF embed):    {smm_full.mean()*100:.1f}% ± {smm_full.std()*100:.1f}%")
print(f"Ablated (Linear embed):  {smm_abl1.mean()*100:.1f}% ± {smm_abl1.std()*100:.1f}%")
print(f"Performance drop: {(smm_full.mean()-smm_abl1.mean())*100:.1f} percentage points")

Full SMM (RBF embed):    89.0% ± 6.4%
Ablated (Linear embed):  54.0% ± 6.0%
Performance drop: 35.0 percentage points


**Interpretation:**
Replacing the RBF embedding kernel with a linear kernel causes a dramatic 35-percentage-point drop in accuracy (89.0% → 54.0%), nearly reaching chance level. This strongly confirms that the RBF embedding kernel is the critical component responsible for the SMM's success on this dataset. The linear kernel collapses the expected kernel to a simple mean-dot-product, making it blind to covariance information — exactly the problem that motivates the SMM in the first place. The size of this drop exceeds my expectation and confirms that in a pure covariance-separation setting, the choice of embedding kernel entirely determines the algorithm's effectiveness. This directly validates the paper's claim (Section 4, Table 2) that "embedding kernels tend to have more impact on the predictive performance compared to the level-2 kernels."


## Component 2: Level-2 Kernel (RBF SVM → Linear SVM)

**Role in the full method:**
The level-2 kernel K operates on the distribution space and defines the shape of the decision boundary between classes of distributions. Using an RBF level-2 kernel (i.e., K(P,Q) = exp(-γ₂‖μ_P - μ_Q‖²)) allows a nonlinear boundary in the space of mean embeddings. Replacing it with a linear SVM constrains the classifier to find only a linear hyperplane separating distributions in the embedding space.


In [ ]:
smm_abl2 = run_smm(distributions, labels, gamma=0.5, kernel='rbf', level2='linear')

print(f"Full SMM (RBF level-2):  {smm_full.mean()*100:.1f}% ± {smm_full.std()*100:.1f}%")
print(f"Ablated (Linear lvl-2):  {smm_abl2.mean()*100:.1f}% ± {smm_abl2.std()*100:.1f}%")
print(f"Performance drop: {(smm_full.mean()-smm_abl2.mean())*100:.1f} percentage points")

Full SMM (RBF level-2):  89.0% ± 6.4%
Ablated (Linear lvl-2):  86.5% ± 5.8%
Performance drop: 2.5 percentage points


**Interpretation:**
Removing the nonlinear level-2 kernel (replacing RBF SVM with linear SVM) causes only a modest 2.5-percentage-point drop (89.0% → 86.5%), well within the standard deviation. This suggests that on our 2D dataset, the decision boundary in embedding space is approximately linear — the two covariance classes form fairly well-separated clusters after the RBF mean embedding. The small drop aligns with the paper's observation (Table 2) that the embedding kernel matters more than the level-2 kernel. This makes intuitive sense: the embedding kernel determines *what information* is captured from each distribution, while the level-2 kernel determines the *boundary shape* — and in this case, linear is sufficient. If the dataset had more complex, overlapping embedding-space structure (as might occur in higher dimensions), the nonlinear level-2 kernel would contribute more.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Task 3.1 — Ablation Study Results', fontsize=12, fontweight='bold')

for ax, comp_label, ablated_val, ablated_std, full_val, full_std, ablated_name, col in zip(
    axes,
    ['Component 1: Embedding Kernel', 'Component 2: Level-2 Kernel'],
    [smm_abl1.mean()*100, smm_abl2.mean()*100],
    [smm_abl1.std()*100,  smm_abl2.std()*100],
    [smm_full.mean()*100, smm_full.mean()*100],
    [smm_full.std()*100,  smm_full.std()*100],
    ['Linear embed\n(→ mean dot product)', 'Linear SVM\n(linear boundary)'],
    ['#E74C3C', '#F39C12']
):
    methods_ab = ['Full SMM\n(RBF embed)', ablated_name]
    vals  = [full_val, ablated_val]
    stds  = [full_std, ablated_std]
    bars = ax.bar(methods_ab, vals, yerr=stds, capsize=7,
                  color=['#27AE60', col], edgecolor='black', lw=1.2, width=0.45, alpha=0.85)
    for bar, v, s in zip(bars, vals, stds):
        ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+s+0.5,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Chance')
    ax.set_ylim(0, 110); ax.set_ylabel('Accuracy (%)'); ax.set_title(comp_label, fontsize=11, fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.35)

plt.tight_layout()
plt.savefig('partB/results/task3_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: partB/results/task3_ablation.png")

Saved: partB/results/task3_ablation.png
